# TAZ-level evaluation (ON_FOOT surrogate)

Evaluates a trained `model_architecture.GCNResNet` checkpoint (from `scripts/train_onfoot.py`)
on its held-out test split, at TAZ resolution. Unlike the old (`qol_surrogate.evaluate`)
notebook, there's no separate hex-resolution or dry-baseline loading step here - this
model bundles its own dry baseline, static features, and graph as buffers, so the
checkpoint is fully self-contained (see `qol_surrogate.model_architecture`).

Metrics come from `qol_surrogate.evaluate_onfoot`, the onfoot analogue of
`qol_surrogate.evaluate` (kept separate since the old module is hardcoded to the
3-mode/21-channel Y_COLS and would mis-index against this model's 7-channel,
POI-category-only output).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch_geometric.loader import DataLoader

from qol_surrogate.data_onfoot import (
    ZONES_FILE, STATIC_FEATURES_DIR, TAZ_PARQUET_DIR, DRY_BASELINE_DIR,
    load_hexes, build_taz_geometries, build_taz_to_idx,
    load_edge_taz_ids, load_dataset_tensors, split_dataset, normalize, build_data_list,
)
from qol_surrogate.evaluate_onfoot import evaluate_onfoot, evaluate_per_taz_onfoot
from qol_surrogate.inference import load_model

In [ ]:
MODELS_DIR = "../models"

# picks the most recently trained onfoot run - point this at a specific folder instead to load an older run
run_dirs = sorted(d for d in glob.glob(os.path.join(MODELS_DIR, "gcn_resnet_onfoot_*")) if os.path.isdir(d))
run_dir = run_dirs[-1]
run_dir

In [ ]:
model, taz_ids = load_model(os.path.join(run_dir, "model.pt"))
model.eval()
len(taz_ids)

## Build the test loader

Re-derives the same train/val/test split `train_onfoot.py` used (`split_dataset`'s
`random_state=13` is fixed), sourced from the same cached `static_features.pkl` /
`graph.pkl` / parquet files `scripts/agg_and_static_features.py` produced - so
`test_loader` here lines up with exactly the scenarios this checkpoint was never
trained on.

In [ ]:
import pickle

with open(os.path.join(STATIC_FEATURES_DIR, "static_features.pkl"), "rb") as f:
    static_features_dict = pickle.load(f)

with open(os.path.join(STATIC_FEATURES_DIR, "graph.pkl"), "rb") as f:
    graph = pickle.load(f)
edge_index, edge_weight, taz_ids = graph["edge_index"], graph["edge_weight"], graph["taz_ids"]

x, y, y_dry = load_dataset_tensors(static_features_dict, taz_parquet_dir=TAZ_PARQUET_DIR,
                                    dry_baseline_dir=DRY_BASELINE_DIR)
x_train, x_val, x_test, y_train, y_val, y_test, _, _, idx_test = split_dataset(x, y)

# normalize with the model's own saved stats (== fresh train-split stats here,
# since split_dataset's random_state is fixed - but using the model's buffers
# directly is the more direct "this is what the checkpoint actually expects" path)
x_test = normalize(x_test, model.x_mean, model.x_std)
y_test = normalize(y_test, model.y_mean, model.y_std)

test_data = build_data_list(x_test, y_test, edge_index, edge_weight)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)  # must stay unshuffled

len(test_loader.dataset)

## Overall (pooled) metrics

`deviation`: how well the model predicts the flood-induced *change* in ON_FOOT
accessibility - what it's actually trained to predict.
`absolute`: predicted deviation + dry baseline vs. true absolute accessibility -
the more real-world-interpretable number.

In [ ]:
results = evaluate_onfoot(model, test_loader)
results["deviation"]["overall"], results["absolute"]["overall"]

## Per POI-category breakdown

In [ ]:
deviation_table = pd.DataFrame(results["deviation"]["per_channel"]).T
deviation_table

In [ ]:
absolute_table = pd.DataFrame(results["absolute"]["per_channel"]).T
absolute_table

## Which TAZ zones have the highest error?

Everything above pools across all TAZ zones into one number per channel. This
computes per-TAZ R2, MAE, C-index, and WAPE (absolute accessibility, pooled across
the 7 POI categories and every test scenario), then maps each - showing *where*
geographically the model struggles most.

In [ ]:
per_taz_metrics = evaluate_per_taz_onfoot(model, test_loader, taz_ids)
# {metric: DataFrame(taz_id -> value, column "ON_FOOT")}
per_taz_metrics["r2"].describe()

In [ ]:
hexes = load_hexes(ZONES_FILE)
tazes = build_taz_geometries(hexes)

In [ ]:
METRIC_LABELS = {'r2': 'R\u00b2', 'wape': 'WAPE', 'c_index': 'C-index'}
# higher-is-better metrics: green=high=good, red=low=bad. WAPE is lower-is-better,
# so its colormap is reversed - green=low=good, red=high=bad
METRIC_CMAPS = {'r2': 'RdYlGn', 'c_index': 'RdYlGn', 'wape': 'RdYlGn_r'}


def plot_per_taz_map(metric):
    df = tazes.join(per_taz_metrics[metric])

    fig, ax = plt.subplots(figsize=(6, 6))
    df.plot(column="ON_FOOT", cmap=METRIC_CMAPS[metric], legend=True, ax=ax,
            legend_kwds={'label': METRIC_LABELS[metric], 'shrink': 0.65})
    ax.set_title(f'Per-TAZ {METRIC_LABELS[metric]} (ON_FOOT, absolute accessibility)')
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


plot_per_taz_map("r2")

In [ ]:
plot_per_taz_map("wape")

In [ ]:
plot_per_taz_map("c_index")